# Ensemble Diversity — Appendix

메인 노트북 `ensemble_diversity_analysis.ipynb` 에서 분리된 부록 표.

- A. Real vs random variance (sanity check)
- B. Region별 `energy_std` / `gar` / `grad_var` 분위수

> EBM energy landscape 2D 시각화는 메인 노트북 섹션 D 에 있음.

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
os.chdir('/home/work/JooKyung/TabEBM')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.precision', 4)

REGION_ORDER = ['real', 'near', 'far']
REGION_COLORS = {'real': '#4E79A7', 'near': '#F28E2B', 'far': '#E15759', 'random': '#76B7B2'}

dist_stats = pd.read_csv('experiments/ebms/stock_distance/compare/ensemble_stats.csv')
sub_stats  = pd.read_csv('experiments/ebms/stock_subsample/compare/ensemble_stats.csv')

---
## A. Real vs Random variance (sanity check)

`real` (class 0 실제 샘플) vs `random` (feature-wise uniform 내에서 랜덤) 에서 variance 비교.
feature 간 상관구조를 무시한 매우 쉬운 OOD라 메인 질문과는 거리가 있음 — 그냥 최소 sanity check.

In [2]:
dist_var = pd.read_csv('experiments/ebms/stock_distance/compare/variance_real_vs_random.csv')
sub_var  = pd.read_csv('experiments/ebms/stock_subsample/compare/variance_real_vs_random.csv')

combined_var = pd.concat([
    dist_var.assign(method='distance'),
    sub_var.assign(method='subsample'),
])[['method', 'group', 'n_points', 'variance_mean', 'variance_median', 'variance_max', 'variance_min']]
print(combined_var.to_string(index=False))

   method         group  n_points  variance_mean  variance_median  variance_max  variance_min
 distance     real_data        53         0.0128           0.0121        0.0342        0.0045
 distance random_points       100         0.1723           0.1595        0.3989        0.0143
subsample     real_data        53         0.0205           0.0092        0.1233        0.0005
subsample random_points       100         0.0185           0.0160        0.0608        0.0022


---
## B. Region별 분위수 테이블

`real`/`near`/`far` × `{distance, subsample}`에 대해 각 지표의 `min/q25/median/q75/max/mean/std`.

### B.1 `energy_std`

In [3]:
rows = []
for r in REGION_ORDER:
    for method, stats_df in [('distance', dist_stats), ('subsample', sub_stats)]:
        vals = stats_df[stats_df.region == r]['energy_std'].values
        rows.append({
            'region': r, 'method': method, 'n': len(vals),
            'min': vals.min(), 'q25': np.quantile(vals, 0.25),
            'median': np.median(vals), 'q75': np.quantile(vals, 0.75),
            'max': vals.max(), 'mean': vals.mean(), 'std': vals.std(),
        })
print(pd.DataFrame(rows).to_string(index=False))

region    method   n    min    q25  median    q75    max   mean    std
  real  distance  53 0.0670 0.0853  0.1099 0.1227 0.1850 0.1098 0.0268
  real subsample  53 0.0218 0.0551  0.0958 0.1545 0.3512 0.1188 0.0797
  near  distance 200 0.0421 0.1967  0.2724 0.3478 0.5678 0.2794 0.1030
  near subsample 200 0.0100 0.0605  0.0874 0.1274 0.2510 0.0942 0.0499
   far  distance 200 0.0436 0.1039  0.1333 0.1784 0.4396 0.1519 0.0719
   far subsample 200 0.0053 0.0375  0.0494 0.0669 0.1677 0.0552 0.0265


### B.2 `gar` (gradient agreement ratio)

In [4]:
rows = []
for r in REGION_ORDER:
    for method, stats_df in [('distance', dist_stats), ('subsample', sub_stats)]:
        vals = stats_df[stats_df.region == r]['gar'].values
        rows.append({
            'region': r, 'method': method, 'n': len(vals),
            'min': vals.min(), 'q25': np.quantile(vals, 0.25),
            'median': np.median(vals), 'q75': np.quantile(vals, 0.75),
            'max': vals.max(), 'mean': vals.mean(), 'std': vals.std(),
        })
print(pd.DataFrame(rows).to_string(index=False))

region    method   n    min    q25  median    q75    max   mean    std
  real  distance  53 0.3325 0.5631  0.6527 0.7361 0.8899 0.6387 0.1252
  real subsample  53 0.3627 0.5803  0.7136 0.8117 0.9581 0.6934 0.1454
  near  distance 200 0.3901 0.6404  0.7366 0.8163 0.9150 0.7260 0.1104
  near subsample 200 0.3542 0.7773  0.8414 0.9027 0.9764 0.8152 0.1250
   far  distance 200 0.1345 0.4999  0.6119 0.7004 0.9079 0.5998 0.1497
   far subsample 200 0.0908 0.6695  0.7969 0.8600 0.9540 0.7359 0.1796


### B.3 `grad_var` (member-간 ∂E/∂x 분산)

> 이 지표는 VP-SGLD (다음 단계) 의 preconditioner 후보로 쓰기 위한 것. 지금 노트북에서는 분산의 존재 여부만 본다.

In [5]:
rows = []
for r in REGION_ORDER:
    for method, stats_df in [('distance', dist_stats), ('subsample', sub_stats)]:
        vals = stats_df[stats_df.region == r]['grad_var'].values
        rows.append({
            'region': r, 'method': method, 'n': len(vals),
            'n_zero': int((vals == 0).sum()),
            'min_nonzero': vals[vals > 0].min() if (vals > 0).any() else np.nan,
            'median': np.median(vals), 'max': vals.max(), 'mean': vals.mean(),
        })
print(pd.DataFrame(rows).to_string(index=False))

region    method   n  n_zero  min_nonzero     median        max       mean
  real  distance  53       0   1.0000e-05 5.0000e-05 8.0700e-04 9.6755e-05
  real subsample  53       0   1.1000e-05 8.5000e-05 5.6100e-04 1.2392e-04
  near  distance 200       0   4.1000e-05 1.8150e-04 9.4200e-04 2.4359e-04
  near subsample 200       0   1.2000e-05 1.0200e-04 7.0700e-04 1.2917e-04
   far  distance 200      10   1.0000e-06 4.0000e-06 9.5000e-05 9.2350e-06
   far subsample 200      19   1.0000e-06 2.0000e-06 6.7000e-05 3.0150e-06
